# Excel (2026년 최신 권장 사용법)

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `UnstructuredExcelLoader(mode="elements")` | **`langchain-unstructured`** 의 `UnstructuredLoader` (시트별 Table 요소 + `text_as_html`) |
| `pd.read_excel` + `DataFrameLoader` | `pd.read_excel` + 직접 `Document` 변환 (시트 단위 Markdown 변환 예시 추가) |

`UnstructuredLoader` 는 `.xlsx` 및 `.xls` 파일 모두에서 작동합니다.

각 시트가 하나의 표(Table) 요소가 되며, 문서 메타데이터의 `text_as_html` 키 아래에 **HTML 표현**이 제공됩니다.

In [ ]:
# 설치
# !pip install -qU langchain-unstructured "unstructured[xlsx]" openpyxl pandas tabulate

In [ ]:
from langchain_unstructured import UnstructuredLoader

# UnstructuredLoader 생성 (파일 형식 자동 감지, 요소 단위 반환)
loader = UnstructuredLoader("./data/titanic.xlsx")

# 문서 로드
docs = loader.load()

# 문서 길이 출력
print(len(docs))

시트 수(및 요소 분할 결과)에 따라 문서 수가 결정됩니다.

`page_content` 에는 표의 텍스트가, `metadata` 의 `text_as_html` 에는 표를 HTML 형식으로 저장합니다.

In [ ]:
# 문서 출력
print(docs[0].page_content[:200])

In [ ]:
# metadata 의 text_as_html 출력
print(docs[0].metadata["text_as_html"][:1000])

In [ ]:
# 어떤 메타데이터가 있는지 확인 (page_name = 시트 이름 등)
{k: v for k, v in docs[0].metadata.items() if k != "text_as_html"}

## pandas 로 직접 변환

### 1) 행 단위 Document (구 `DataFrameLoader`)

CSV 와 마찬가지로 `read_excel()` 로 DataFrame 을 만든 뒤 `Document` 로 변환합니다.

In [ ]:
import pandas as pd

# Excel 파일 읽기 (첫 번째 시트)
df = pd.read_excel("./data/titanic.xlsx")

In [ ]:
from typing import Iterator

from langchain_core.documents import Document


def dataframe_to_documents(df: pd.DataFrame, page_content_column: str) -> Iterator[Document]:
    """한 컬럼은 본문, 나머지 컬럼은 메타데이터로 변환"""
    meta_df = df.drop(columns=[page_content_column]).astype(object)
    meta_df = meta_df.where(pd.notna(meta_df), None)  # NaN → None
    for content, metadata in zip(df[page_content_column], meta_df.to_dict(orient="records")):
        yield Document(page_content=str(content), metadata=metadata)


# 문서 로드
docs = list(dataframe_to_documents(df, page_content_column="Name"))

# 데이터 출력
print(docs[0].page_content)

# 메타데이터 출력
print(docs[0].metadata)

### 2) 시트 단위 Document (Markdown 표)

LLM 은 HTML 보다 **Markdown 표**를 더 적은 토큰으로 잘 이해하는 경우가 많습니다. 여러 시트를 한 번에 읽어 시트별 문서로 만들 수 있습니다. (`to_markdown()` 은 `tabulate` 패키지가 필요합니다)

In [ ]:
sheets = pd.read_excel("./data/titanic.xlsx", sheet_name=None)  # {시트명: DataFrame}

sheet_docs = [
    Document(
        page_content=sheet_df.to_markdown(index=False),
        metadata={"source": "./data/titanic.xlsx", "sheet": name, "rows": len(sheet_df)},
    )
    for name, sheet_df in sheets.items()
]

print(len(sheet_docs))
print(sheet_docs[0].page_content[:500])

> 표가 매우 크면 한 문서가 지나치게 길어집니다. 이런 경우 N행씩 잘라서(헤더를 매 조각마다 반복) 여러 문서로 나누는 것이 검색에 유리합니다.

In [ ]:
def dataframe_to_markdown_chunks(df: pd.DataFrame, rows_per_chunk: int = 50, **metadata):
    for start in range(0, len(df), rows_per_chunk):
        part = df.iloc[start : start + rows_per_chunk]
        yield Document(
            page_content=part.to_markdown(index=False),  # 조각마다 헤더 포함
            metadata={**metadata, "row_start": start, "row_end": start + len(part) - 1},
        )


chunks = list(dataframe_to_markdown_chunks(df, 50, source="./data/titanic.xlsx"))
print(len(chunks))
print(chunks[0].metadata)